# Lab 2 — Déploiement local avec Ollama

**Cours 17 — Environnements, Infrastructures & Architectures de Déploiement**  
MSc IAG — AIvancity | Certificat 3

---

## Objectifs
- Installer et configurer Ollama sur une VM Vertex AI Workbench
- Télécharger et exécuter des modèles open-source (Llama 3.2, Gemma 3)
- Comprendre la quantization GGUF en pratique (INT4, INT8, FP16)
- Comparer latence et qualité entre Ollama (local) et Gemini (cloud)
- Identifier les cas d'usage où l'on-premise est préférable

## Prérequis
- VM Vertex AI Workbench avec GPU (recommandé : T4 ou L4) ou CPU
- Accès terminal à la VM

**Durée estimée :** 1h


## 0. Installation d'Ollama

Ouvrez un **Terminal**  et exécutez :

```bash
# 1. Installer Ollama
curl -fsSL https://ollama.com/install.sh | sh

# 2. Démarrer le serveur Ollama en arrière-plan
ollama serve &

# 3. Vérifier la version
ollama --version

# 4. Télécharger les modèles (patience - quelques minutes)
ollama pull llama3.2:3b        # ~2 GB  — modèle rapide pour exercices
ollama pull gemma3:4b          # ~2.5 GB — modèle Google open-source
# ollama pull llama3.3:70b-q4  # ~40 GB — si GPU disponible (optionnel)

# 5. Vérifier les modèles disponibles
ollama list
```


Attendez que les modèles soient téléchargés avant de continuer. Le serveur écoute sur `http://localhost:11434`.

In [ ]:
# Vérification que le serveur Ollama est actif
import requests, json

BASE_URL = 'http://localhost:11434'

def check_ollama():
    try:
        r = requests.get(BASE_URL, timeout=5)
        return r.status_code == 200
    except Exception as e:
        return False

def list_models():
    try:
        r = requests.get(f'{BASE_URL}/api/tags', timeout=10)
        return [m['name'] for m in r.json().get('models', [])]
    except Exception as e:
        return [f'Erreur: {e}']

if check_ollama():
    print('✅ Serveur Ollama actif')
    models = list_models()
    print(f'Modèles disponibles : {models}')
else:
    print('❌ Serveur Ollama non accessible — exécutez `ollama serve &` dans le terminal')


---
## Partie 1 — Premier appel avec Ollama


In [ ]:
!pip install -q ollama


In [ ]:
from ollama import Client
import time

client = Client(host=BASE_URL)

# Choisissez votre modèle (modifiez selon ce qui est disponible dans ollama list)
MODEL_LOCAL = 'llama3.2:3b'  # ou 'gemma3:4b'

def chat(prompt, model=MODEL_LOCAL, system=None):
    """Appel simple à Ollama."""
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    
    start = time.time()
    resp = client.chat(model=model, messages=messages,
                       options={'temperature': 0.7, 'num_ctx': 4096})
    elapsed = time.time() - start
    return resp['message']['content'], elapsed

# Test
answer, t = chat('Explique en 2 phrases pourquoi les LLMs open-source sont importants pour la souveraineté des données.')
print(f'Réponse ({t:.1f}s) :\n{answer}')


In [ ]:
# Streaming — plus agréable pour les démos
print('Streaming en temps réel :')
stream = client.chat(
    model=MODEL_LOCAL,
    messages=[{'role': 'user', 'content': 'Liste 5 avantages du déploiement on-premise pour les LLMs en entreprise.'}],
    stream=True
)
for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)
print()


---
## Partie 2 — Quantization en pratique

Ollama utilise le format **GGUF** (GPT-Generated Unified Format) développé par llama.cpp.  
Différents niveaux de quantization sont disponibles pour chaque modèle.  

| Format | Bits | Taille (7B) | Vitesse | Qualité |
|--------|------|-------------|---------|--------|
| Q4_K_M | 4    | ~4 GB       | 4-6×    | -2-3% perf |
| Q8_0   | 8    | ~7 GB       | 2-3×    | -0.5% perf |
| F16    | 16   | ~14 GB      | 1.5×    | Référence |

Les tags Ollama comme `llama3.2:3b` utilisent Q4_K_M par défaut.

In [ ]:
# Comparer deux niveaux de quantization (si disponibles)
# Note: sur CPU, INT4 est souvent 2-3× plus rapide que INT8

test_prompt = (
    'Calcule le coût mensuel d\'une API LLM si on fait 100 000 requêtes par jour '
    'avec 500 tokens en entrée et 200 tokens en sortie, au tarif de $0.075 par million de tokens input '
    'et $0.30 par million de tokens output. Montre le calcul détaillé.'
)

def benchmark(prompt, model, label):
    start = time.time()
    resp = client.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1}
    )
    elapsed = time.time() - start
    answer = resp['message']['content']
    words = len(answer.split())
    print(f'\n── {label} ──────────────────────')
    print(f'Latence : {elapsed:.2f}s | Mots générés : {words}')
    print(f'Réponse : {answer[:400]}...' if len(answer) > 400 else f'Réponse : {answer}')

benchmark(test_prompt, MODEL_LOCAL, MODEL_LOCAL)
# Si vous avez un second modèle disponible :
# benchmark(test_prompt, 'gemma3:4b', 'Gemma3 4B')


In [ ]:
# ── TODO 2.1 ─────────────────────────────────────────────────────────────
# Téléchargez un second modèle (ex: gemma3:4b ou mistral:7b) via le terminal :
#   ollama pull gemma3:4b
# Puis comparez les deux modèles sur le même prompt de raisonnement.
# Documentez : latence, qualité perçue, nb de tokens/s approximatif.

# VOTRE CODE ICI


---
## Partie 3 — Embeddings locaux avec Ollama

Les embeddings locaux permettent d'éviter d'envoyer des données sensibles vers des APIs externes.

In [ ]:
import numpy as np

def embed(text, model=MODEL_LOCAL):
    """Génère un embedding via Ollama."""
    resp = client.embeddings(model=model, prompt=text)
    return np.array(resp['embedding'])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Base de connaissances mini
docs = [
    'Gemini 2.5 Flash coûte $0.075 par million de tokens en entrée.',
    'La quantization INT4 réduit la taille du modèle de 75% avec une perte de qualité inférieure à 3%.',
    'Ollama permet de déployer des LLMs open-source localement sans GPU dédié.',
    'Le semantic caching peut réduire les appels API de 40 à 60% sur des workloads conversationnels.',
    'LiteLLM est un proxy unifié compatible avec 100+ LLMs via une API OpenAI-compatible.',
]

print('Calcul des embeddings (peut prendre 30-60s)...')
doc_embeddings = [embed(d) for d in docs]
print(f'✅ {len(doc_embeddings)} embeddings générés, dimension: {len(doc_embeddings[0])}')


In [ ]:
# Mini-RAG : retrieval par similarité cosinus
def retrieve(query, k=2):
    q_emb = embed(query)
    scores = [(cosine_similarity(q_emb, de), doc) for de, doc in zip(doc_embeddings, docs)]
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:k]

queries = [
    'Quel est le prix de Gemini Flash ?',
    'Comment réduire les coûts de déploiement LLM ?',
    'Peut-on faire du RAG sans Cloud ?'
]

for q in queries:
    print(f'\nQuery : "{q}"')
    results = retrieve(q, k=2)
    for score, doc in results:
        print(f'  [{score:.3f}] {doc}')


---
## Partie 4 — Comparaison Ollama vs Gemini API

Quand utiliser l'une ou l'autre solution ?

In [ ]:
# Vous avez besoin du client Vertex AI du Lab 1
# Décommentez si vous n'avez pas encore initialisé le client Vertex
# from google import genai
# client_gcp = genai.Client(vertexai=True, project='your-project-id', location='us-central1')

import time

benchmark_prompt = (
    'Explique en 5 points concrets comment choisir entre un déploiement '
    'cloud (Vertex AI) et on-premise (Ollama) pour un LLM en entreprise. '
    'Inclus des critères quantitatifs (coût, latence, sécurité).'
)

# -- Ollama local --
start = time.time()
resp_local = client.chat(
    model=MODEL_LOCAL,
    messages=[{'role': 'user', 'content': benchmark_prompt}],
    options={'temperature': 0.5}
)['message']['content']
t_local = time.time() - start

print('=== OLLAMA (local) ===')
print(f'Latence : {t_local:.2f}s')
print(resp_local[:600])


In [ ]:
# -- Gemini Flash (cloud) -- (décommentez si client Vertex AI disponible)
# start = time.time()
# resp_cloud = client_gcp.models.generate_content(
# model='gemini-2.5-flash',
#     contents=benchmark_prompt
# )
# t_cloud = time.time() - start
# print('=== GEMINI FLASH (cloud) ===')
# print(f'Latence : {t_cloud:.2f}s')
# print(resp_cloud.text[:600])

# RÉSUMÉ COMPARATIF — remplissez après vos tests
comparison = {
    'Ollama Llama3.2 3B':  {'latence_s': None, 'coût_requête': '$0 (infra fixe)', 'données_locales': True,  'gpu_requis': False},
    'Gemini 2.5 Flash':    {'latence_s': None, 'coût_requête': '$0.00015/req*', 'données_locales': False, 'gpu_requis': False},
}
# (*) estimation 500 input + 200 output tokens
print('\nTableau comparatif à compléter après vos tests :')
for model, vals in comparison.items():
    print(f'  {model}: {vals}')


In [ ]:
# ── TODO 4 ───────────────────────────────────────────────────────────────
# 1. Testez 3 types de requêtes différentes (simple, complexe, longue)
#    et mesurez la latence pour Ollama.
# 2. Identifiez à partir de quelle complexité Ollama décroche en qualité.
# 3. Pour quels cas d'usage de votre stage/entreprise utiliseriez-vous Ollama ?
#    Argumentez avec les critères vus en cours (données, coût, latence, souveraineté).

# VOTRE CODE ICI
